# L4 单层 cache-path · 当前分数与 score EMA
选择 NVIDIA L4，点击「全部运行」。使用已完成的 KV_L4_Short_Heldout_Quest_results.zip（约 264 MiB）。如果已将它放在 /content，则直接读取，否则弹窗选择文件。输入文件通过 SHA-256 校验。

40 篇、每篇 28 层、B=128、K=64、EMA α=0.5。起点是本层当前步的 top-64 和 FP32 Quest 分数已在 GPU，计时覆盖 resolver、miss 描述符回传和同步、主机打包、pinned H2D、GPU 槽位写入和完成同步。模型推理和产生 Quest 分数的时间在该测量范围之外。分数策略使用相同的候选排序和平局规则。EMA 用 FP64 更新归一化分数以匹配已有的 CPU 参考精度，计时对应这份实现。

In [ ]:
import json,os,subprocess,sys,tempfile,shutil
from pathlib import Path
from google.colab import files
import torch
assert torch.cuda.is_available() and "L4" in torch.cuda.get_device_name().split(),"请选择 NVIDIA L4"
source=Path("/content/KV_L4_Short_Heldout_Quest_results.zip")
if not source.exists():
    payload=files.upload()
    assert len(payload)==1,"请选择 held-out 分数结果包"
    source=Path(next(iter(payload))).resolve()
    del payload
work=Path(tempfile.mkdtemp(prefix="kv-score-hw-",dir="/content"))
os.chdir(work)
out=work/"results"
out.mkdir()
(out/"gpu.txt").write_text(subprocess.check_output(["nvidia-smi"],text=True))
print("输入",source,"输出",out)

### 实现与 CPU 参考
每轮从空缓存开始。三种策略共用 64 KiB 合成 BF16 页与相同的实际搬运路径。四个任务 × 长度组各取一个存在策略差异的层做预热与逐步页内容检查。

In [ ]:
%%writefile sim.py
import heapq
import numpy as np

class Policy:
    """Base: LRU bookkeeping. evict() returns n victims from candidates (unselected residents)."""
    needs_scores = False

    def __init__(self, K, **kw):
        self.K = K
        self.last = {}
        self.hit = {}
        self.params = kw

    def begin(self, t, ranked, srow):
        self.t, self.ranked, self.srow = t, ranked, srow

    def evict(self, cands, n):
        return heapq.nsmallest(n, cands, key=self.key)

    def key(self, p):
        return self.last[p], self.hit[p], -p

    def dropped(self, p):
        self.last.pop(p, None)
        self.hit.pop(p, None)

    def admit(self, hits, misses):
        for p in self.ranked:
            self.last[p] = self.t
            self.hit[p] = p in hits

class LRU(Policy):
    """tie: how pages last selected at the same step are ordered (key ascending = evicted first)
       "miss_first" (default, as replay.py): among equals, pages that were misses leave first
       "hit_first": pages that were hits leave first (newly loaded pages are protected)
       "id": larger page ID leaves first"""
    def __init__(self, K, tie="miss_first", **kw):
        super().__init__(K, **kw)
        if tie not in ("miss_first", "hit_first", "id"):
            raise ValueError("bad tie")
        self.tie = tie

    def key(self, p):
        if self.tie == "miss_first":
            return self.last[p], self.hit[p], -p
        if self.tie == "hit_first":
            return self.last[p], not self.hit[p], -p
        return self.last[p], -p

def _z(srow, ranked, p):
    """Score of p relative to the K-th selected score, in units of (top1 - K-th)."""
    kth, top = srow[ranked[-1]], srow[ranked[0]]
    return (srow[p] - kth) / (top - kth + 1e-6)

class ScoreCurrent(Policy):
    """Evict the lowest current selector score (candidates nearest to re-selection stay)."""
    needs_scores = True

    def key(self, p):
        return self.srow[p], self.last[p], self.hit[p], -p

class ScoreEMA(Policy):
    """Evict the lowest exponential moving average of a per-step page signal (one float per resident).

    signal: "z"    score relative to the K-th selected score, in units of (top1 - K-th)
            "raw"  the raw selector score
            "rank" rank-only information (schema v1): (K - rank + 1) / K if selected, else 0
    mode:   "all"       every resident is updated every step (needs every resident's current score)
            "selected"  a resident is updated only on steps where it is selected
    """
    needs_scores = True

    def __init__(self, K, alpha=0.5, mode="all", signal="z", **kw):
        super().__init__(K, **kw)
        if mode not in ("all", "selected") or signal not in ("z", "raw", "rank"):
            raise ValueError("bad mode or signal")
        self.alpha, self.mode, self.signal = alpha, mode, signal
        self.ema = {}

    def value(self, p):
        if self.signal == "z":
            return _z(self.srow, self.ranked, p)
        if self.signal == "raw":
            return self.srow[p]
        r = self.rank_of.get(p)
        return 0.0 if r is None else (self.K - r + 1) / self.K

    def begin(self, t, ranked, srow):
        super().begin(t, ranked, srow)
        self.rank_of = {p: r for r, p in enumerate(ranked, 1)}
        a = self.alpha
        pages = self.ema if self.mode == "all" else [p for p in ranked if p in self.ema]
        for p in pages:
            self.ema[p] = a * self.ema[p] + (1 - a) * self.value(p)

    def key(self, p):
        return self.ema[p], self.last[p], self.hit[p], -p

    def dropped(self, p):
        super().dropped(p)
        del self.ema[p]

    def admit(self, hits, misses):
        super().admit(hits, misses)
        for p in misses:
            self.ema[p] = self.value(p)

def simulate(ranked, scores, B, name, *, on_evict=None, miss_log=None, **params):
    """Replay one layer stream. ranked: [T, K] ints; scores: [T, P] or None.
    Returns dict(accesses, misses, compulsory). on_evict(t, victims, policy) is optional."""
    rows = [tuple(int(p) for p in r if p >= 0) for r in np.asarray(ranked)]
    K = max(len(r) for r in rows)
    if B < K:
        raise ValueError("budget below K")
    cls = POLICIES[name]
    pol = cls(K, **params)
    if cls.needs_scores and scores is None:
        raise ValueError(f"{name} needs scores")
    res, seen = set(), set()
    acc = miss = 0
    for t, row in enumerate(rows):
        srow = scores[t].tolist() if cls.needs_scores else None
        pol.begin(t, row, srow)
        sel = set(row)
        hits = sel & res
        misses = sel - hits
        if miss_log is not None:
            miss_log.append(len(misses))
        n = len(res) + len(misses) - B
        if n > 0:
            victims = pol.evict(res - sel, n)
            if on_evict is not None:
                on_evict(t, victims, pol)
            for v in victims:
                res.discard(v)
                pol.dropped(v)
        res |= sel
        pol.admit(hits, misses)
        assert len(res) <= B
        acc += len(sel)
        miss += len(misses)
        seen |= sel
    return dict(accesses=acc, misses=miss, compulsory=len(seen))

POLICIES = {"lru":LRU,"score":ScoreCurrent,"ema":ScoreEMA}

In [ ]:
%%writefile benchmark.py
import csv,gzip,io,itertools,json,hashlib,sys
from collections import defaultdict
from pathlib import Path
from zipfile import ZipFile
from time import perf_counter_ns
import numpy as np
from sim import simulate
def file_hash(path):
    with path.open("rb") as f: return hashlib.file_digest(f,"sha256").hexdigest()
EXPECTED_SHA='cbc9ccc394f329c6f26cbac2474ab825a50bd40add6545ac805affd07437ca1f'
import torch
import triton
import triton.language as tl

BUDGET, TOP_K = 128, 64
PAGE_SHAPE = (16,2,8,128)
PAGE_BYTES = 65536
NAMES = ("LRU","current score","score EMA")

@triton.jit(do_not_specialize=["step"])
def resolve(Query, Scores, SlotPage, PageSlot, Last, WasHit, Ema, Descriptor, Dest,
            step, B:tl.constexpr, K:tl.constexpr, MODE:tl.constexpr):
    lane=tl.arange(0,B)
    q=lane<K
    wanted=tl.load(Query+lane,q,other=0)
    old_slot=tl.load(PageSlot+wanted,q,other=-1)
    hit=q & (old_slot>=0)
    miss=q & ~hit
    bins=tl.histogram(tl.where(hit,old_slot,0),B)
    protected=bins>tl.where(lane==0,tl.sum((~hit).to(tl.int32),0),0)
    page=tl.load(SlotPage+lane)
    used=page>=0
    safe=tl.maximum(page,0)
    last=tl.load(Last+safe,used,other=0)
    was_hit=tl.load(WasHit+safe,used,other=0)
    # Lower score loses; ties use (last, was_hit, -page_id). Slot breaks empty-slot ties.
    # Short prompts have < 2048 pages and at most 127 decode steps.
    tie=(((last*2+was_hit)*2048+2047-safe)*B+lane)
    if MODE==0:
        ordered=tl.sort(tl.where(protected,2147483647,tl.where(used,tie,lane-B)))
        # Protected lanes never serve misses.
        slot_order=ordered & (B-1)
    else:
        v=tl.load(Scores+safe,used,other=0.)
        if MODE==2:
            first=tl.load(Query)
            kth=tl.load(Query+K-1)
            vmax=tl.load(Scores+first).to(tl.float64)
            vmin=tl.load(Scores+kth).to(tl.float64)
            scale=vmax-vmin+1.e-6
            z=(v.to(tl.float64)-vmin)/scale
            prev=tl.load(Ema+safe,used,other=0.)
            v=0.5*prev+0.5*z
            tl.store(Ema+safe,v,used)
            w=tl.load(Scores+wanted,miss,other=0.).to(tl.float64)
            tl.store(Ema+wanted,(w-vmin)/scale,miss)
        # Score plus integer tie (FP32 current score, FP64 EMA). Bitonic sort carries the slot in the low tie bits.
        v=tl.where(protected,float("inf"),tl.where(used,v,-float("inf")))
        for k in tl.static_range(1,8):
            for j in tl.static_range(k):
                stride=1<<(k-1-j)
                partner=lane ^ stride
                pv=tl.gather(v,partner,axis=0)
                pt=tl.gather(tie,partner,axis=0)
                smaller=(pv<v) | ((pv==v)&(pt<tie))
                larger=(pv>v) | ((pv==v)&(pt>tie))
                want_min=((lane & (1<<k))==0)==((lane & stride)==0)
                swap=tl.where(want_min,smaller,larger)
                v=tl.where(swap,pv,v)
                tie=tl.where(swap,pt,tie)
        slot_order=tie & (B-1)
    index=tl.maximum(tl.cumsum(miss.to(tl.int32),0)-1,0)
    dest=tl.gather(slot_order,index,axis=0)
    evicted=tl.gather(page,dest,axis=0)
    tl.store(PageSlot+tl.maximum(evicted,0),-1,miss & (evicted>=0))
    tl.store(PageSlot+wanted,dest,miss)
    tl.store(SlotPage+dest,wanted,miss)
    tl.store(Descriptor+1+index,wanted,miss)
    tl.store(Dest+index,dest,miss)
    tl.store(Descriptor,tl.sum(miss.to(tl.int32),0))
    tl.store(Last+wanted,step,q)
    tl.store(WasHit+wanted,hit.to(tl.int32),q)

class LayerCache:
    def __init__(self, pages):
        import torch

        self.torch = torch
        self.resolve = resolve
        self.pages = pages
        self.stream = torch.cuda.current_stream()
        self.host = torch.empty((pages, *PAGE_SHAPE), dtype=torch.bfloat16, pin_memory=True)
        self.host.uniform_(-1.0, 1.0, generator=torch.Generator().manual_seed(31278))
        self.host_stage = torch.empty((TOP_K, *PAGE_SHAPE), dtype=torch.bfloat16, pin_memory=True)
        self.gpu_stage = torch.empty_like(self.host_stage, device="cuda", pin_memory=False)
        self.slots = torch.empty((BUDGET, *PAGE_SHAPE), device="cuda", dtype=torch.bfloat16)
        self.slot_page = torch.empty(BUDGET, device="cuda", dtype=torch.int32)
        self.page_slot = torch.empty(pages, device="cuda", dtype=torch.int32)
        self.last = torch.empty_like(self.page_slot)
        self.hit = torch.empty_like(self.page_slot)
        self.ema = torch.empty(pages,device="cuda",dtype=torch.float64)
        self.descriptor = torch.empty(TOP_K + 1, device="cuda", dtype=torch.int64)
        self.host_descriptor = torch.empty(TOP_K + 1, dtype=torch.int64, pin_memory=True)
        self.dest = torch.empty(TOP_K, device="cuda", dtype=torch.int64)
        self.copy_views = [(self.host_descriptor[1:1+n], self.host_stage[:n],
                            self.gpu_stage[:n], self.dest[:n]) for n in range(TOP_K + 1)]
        self.events = [torch.cuda.Event(enable_timing=True) for _ in range(4)]

    def reset(self):
        self.slot_page.fill_(-1)
        self.page_slot.fill_(-1)
        self.stream.synchronize()

    def replay(self, queries, scores, mode, *, timing=False, breakdown=False, validate=False):
        torch = self.torch
        self.reset()
        records = []
        policy_start, policy_end, copy_start, copy_end = self.events
        # Materialize Python views before timing so input/rank preparation is out of scope.
        for step, (wanted, score_row) in enumerate(zip(queries.unbind(),scores.unbind(),strict=True)):
            if timing:
                start = perf_counter_ns()
            if breakdown:
                policy_start.record(self.stream)
            self.resolve[(1,)](wanted, score_row, self.slot_page, self.page_slot, self.last, self.hit,
                               self.ema, self.descriptor, self.dest, step,
                               BUDGET, TOP_K, mode, num_warps=4, enable_fp_fusion=False)
            if breakdown:
                policy_end.record(self.stream)
            self.host_descriptor.copy_(self.descriptor, non_blocking=True)
            self.stream.synchronize()
            misses = int(self.host_descriptor[0])
            if not 0 <= misses <= TOP_K:
                raise RuntimeError(f"Resolver emitted invalid miss count {misses} at step {step}")
            pack_us = None
            if misses:
                page_ids, host_stage, gpu_stage, dest = self.copy_views[misses]
                if breakdown:
                    pack_start = perf_counter_ns()
                torch.index_select(self.host, 0, page_ids, out=host_stage)
                if breakdown:
                    pack_us = (perf_counter_ns() - pack_start) / 1000
                    copy_start.record(self.stream)
                gpu_stage.copy_(host_stage, non_blocking=True)
                if breakdown:
                    copy_end.record(self.stream)
                self.slots.index_copy_(0, dest, gpu_stage)
                self.stream.synchronize()
            wall = (perf_counter_ns() - start) / 1000 if timing else None
            policy_us = policy_start.elapsed_time(policy_end) * 1000 if breakdown else None
            copy_us = (copy_start.elapsed_time(copy_end) * 1000 if misses else 0.0) if breakdown else None
            if breakdown and not misses:
                pack_us = 0.0
            if validate:
                wanted_slots = self.page_slot.index_select(0, wanted.to(torch.int64)).to(torch.int64)
                assert bool((wanted_slots >= 0).all().item())
                assert torch.equal(self.slot_page.index_select(0, wanted_slots), wanted)
                reference = self.host.index_select(0, wanted.cpu().to(torch.int64))
                assert torch.equal(self.slots.index_select(0, wanted_slots).cpu().view(torch.int16),
                                   reference.view(torch.int16))
            records.append((misses, wall, policy_us, pack_us, copy_us))
        return records



METRICS=("wall_us","resolver_us","host_pack_us","h2d_us")
def write(path,rows):
    with path.open("w",newline="") as f:
        w=csv.DictWriter(f,rows[0].keys());w.writeheader();w.writerows(rows)

def summarize(rows,out):
    groups=defaultdict(list)
    base=[r for r in rows if r["policy"]==NAMES[0]]
    for i,r in enumerate(base):groups[r["task"],r["band"]].append(i)
    assert sorted(map(len,groups.values()))==[10]*4
    rng=np.random.default_rng(20261119)
    idx=np.concatenate([rng.choice(g,(10000,len(g))) for g in groups.values()],axis=1)
    steps=np.array([r["steps"] for r in base])
    result=[]
    for a,b in [(0,2),(1,2),(0,1)]:
        for m in METRICS:
            u,v=(np.array([r[m] for r in rows if r["policy"]==NAMES[j]]) for j in (a,b))
            diff=v-u
            ci=np.percentile((diff*steps)[idx].sum(1)/steps[idx].sum(1),[2.5,97.5])
            row=dict(baseline=NAMES[a],policy=NAMES[b],metric=m,baseline_mean=np.average(u,weights=steps),
                     policy_mean=np.average(v,weights=steps),delta_us=np.average(diff,weights=steps),
                     ci_low_us=ci[0],ci_high_us=ci[1],faster_docs=int(sum(diff<0)))
            result.append(row)
            if m=="wall_us":
                verdict="faster" if ci[1]<0 else "slower" if ci[0]>0 else "inconclusive"
                print(f"{NAMES[b]} minus {NAMES[a]}: {row['delta_us']:+.3f} us/layer-step, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}] ({verdict})")
    write(out/"paired.csv",result)

def run(bundle,out):
    meta=dict(gpu=torch.cuda.get_device_name(), torch=torch.__version__,cuda=torch.version.cuda,triton=triton.__version__,
              B=128,K=64,alpha=0.5,page_bytes=PAGE_BYTES,wall_repeats=3,breakdown_repeats=1,
              ema_dtype="float64, matches Python reference",inputs="FP32 Quest scores and ranked IDs resident on GPU",
              payload="synthetic BF16 KV, deterministic seed 31278",bootstrap="10000 stratified paired document draws, seed 20261119")
    assert "L4" in meta["gpu"].split()
    meta["trace_zip_sha256"]=file_hash(bundle)
    assert meta["trace_zip_sha256"]==EXPECTED_SHA
    (out/"environment.json").write_text(json.dumps(meta,indent=2))
    z=ZipFile(bundle)
    docs=json.loads(z.read("original_manifest.json"))["heldout"]
    assert len(docs)==len({d["request_id"] for d in docs})==40
    archive={(r["doc"],int(r["layer"]),r["policy"]):r for r in csv.DictReader(io.StringIO(z.read("by_layer.csv").decode()))}
    cache=LayerCache(max((d["prompt_tokens"]+128)//16 for d in docs))
    assert cache.pages<2048
    smoke_done=set()
    def load(i):
        with np.load(io.BytesIO(z.read(f"scores/{i+1:02}.npz")),allow_pickle=False) as f:
            assert str(f["rid"])==docs[i]["request_id"] and int(f["prompt_tokens"])==docs[i]["prompt_tokens"]
            r,s=f["ranked"],f["scores"]
        assert r.shape==(127,28,64) and s.dtype==np.float32
        return r,s
    def prepare(i,l,r,s):
        x=np.ascontiguousarray(r[:,l],dtype=np.int32)
        y=np.ascontiguousarray(s[:,l],dtype=np.float32)
        counts=[]
        for name,pol,kw in zip(NAMES,("lru","score","ema"),({},{},{"alpha":0.5})):
            steps=[]
            c=simulate(x,y,128,pol,miss_log=steps,**kw)
            a=archive[docs[i]["request_id"],l,name]
            assert all(int(a[k])==v for k,v in c.items())
            counts.append(steps)
        # Align GPU row starts consistently across layers and documents.
        aligned=np.pad(y,((0,0),(0,(-y.shape[1])%16)),constant_values=np.nan)
        return torch.from_numpy(x).cuda(),torch.from_numpy(aligned).cuda(),counts
    smoke=[]
    for i,d in enumerate(docs):
        group=(d["request_id"].split("/")[0],d["length_band_start"])
        if group in smoke_done: continue
        r,s=load(i)
        # Pick a layer exercising actual policy differences in this stratum.
        l=max(range(28),key=lambda l:int(archive[d["request_id"],l,"LRU"]["misses"])-int(archive[d["request_id"],l,"score EMA"]["misses"]))
        x,y,refs=prepare(i,l,r,s)
        for mode,name in enumerate(NAMES):
            rows=cache.replay(x,y,mode,validate=True)
            assert [row[0] for row in rows]==refs[mode],(i,l,name,"smoke")
            smoke.append(dict(doc=d["request_id"],layer=l,policy=name,steps=len(rows),misses=sum(refs[mode]),payload_pages_checked=len(rows)*64))
        smoke_done.add(group)
    assert len(smoke_done)==4
    write(out/"smoke.csv",smoke)
    print("Smoke passed: four strata, exact BF16 slots and per-step CPU counts.",flush=True)
    orders=list(itertools.permutations(range(3)))
    totals=np.zeros(3,dtype=np.int64)
    paired=[]
    with gzip.open(out/"timings.csv.gz","wt",newline="") as stream:
        w=csv.writer(stream);w.writerow(("doc","layer","round","order_position","policy","step","misses","h2d_bytes","wall_us","resolver_us","host_pack_us","h2d_us"))
        for i,d in enumerate(docs):
            r,s=load(i)
            # Each doc has 3 wall rounds and a separate instrumented round.
            sums=np.zeros((3,4));steps_total=0
            for l in range(28):
                x,y,refs=prepare(i,l,r,s)
                totals+=np.sum(refs,axis=1)
                steps_total+=len(x)
                for rep in range(4):
                    phases=rep==3
                    # Latin rotation gives each policy each position once in wall rounds.
                    order=orders[(i*28+l)%6]
                    order=order[rep%3:]+order[:rep%3]
                    for pos,mode in enumerate(order):
                        rows=cache.replay(x,y,mode,timing=not phases,breakdown=phases)
                        observed=[row[0] for row in rows]
                        if observed!=refs[mode]:
                            t=next(t for t,(a,b) in enumerate(zip(observed,refs[mode])) if a!=b)
                            raise ValueError((d["request_id"],l,rep,NAMES[mode],t,observed[t],refs[mode][t]))
                        for t,(m,wall,res,pack,copy) in enumerate(rows):
                            w.writerow((d["request_id"],l,rep,pos,NAMES[mode],t,m,m*PAGE_BYTES,wall,res,pack,copy))
                            if phases:sums[mode,1:]+=(res,pack,copy)
                            else:sums[mode,0]+=wall
            for mode,name in enumerate(NAMES):
                means=sums[mode]/(steps_total*np.array([3,1,1,1]))
                paired.append(dict(doc=d["request_id"],task=d["request_id"].split("/")[0],band=d["length_band_start"],
                                   policy=name,steps=steps_total,**dict(zip(METRICS,means))))
            stream.flush()
            write(out/"documents.csv",paired)
            print(f"{i+1}/40 complete",flush=True)
    z.close()
    assert totals.tolist()==[388049,362955,358608],totals
    write(out/"traffic.csv",[dict(policy=n,misses=int(t),repeat_misses=int(t)-225680,h2d_bytes=int(t)*PAGE_BYTES) for n,t in zip(NAMES,totals)])
    summarize(paired,out)

if __name__=="__main__": run(Path(sys.argv[1]),Path(sys.argv[2]))

### 运行 smoke 和完整对照
每个文档–层在每种策略上做 3 次同步 wall 计时，轮换顺序，再做 1 次独立分项计时（CUDA events 测 resolver 和 H2D，CPU 时钟测 host pack）。每轮逐步核对 GPU miss 与 CPU，逐层核对输入包计数。预期 LRU、当前分数、score EMA 的重复 miss 分别为 162,369、137,275、132,928。

主判据是配对 wall 差（策略减基线），单位 µs/层/步。区间完全低于零表示净加速，完全高于零表示净减速。分项均值来自独立测量轮，适合解释耗时变化。

In [ ]:
if (out/"run.log").exists(): raise RuntimeError("请从第一个代码格开始新一轮测量。")
with (out/"run.log").open("w") as log, subprocess.Popen(
        [sys.executable,"-u","benchmark.py",str(source),str(out)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as proc:
    for line in proc.stdout:
        print(line,end="",flush=True)
        log.write(line)
    rc=proc.wait()
done=rc==0 and (out/"paired.csv").is_file()
(out/"status.json").write_text(json.dumps({"complete":done,"returncode":rc}))
for n in ("benchmark.py","sim.py"):shutil.copy2(n,out/n)
name="KV_L4_Score_Cache_Path_"+("results" if done else "diagnostics")
result_zip=shutil.make_archive(str(work/name),"zip",out)
if done:
    import pandas as pd
    display(pd.read_csv(out/"traffic.csv"))
    display(pd.read_csv(out/"paired.csv").round(4))
else: print("实验未通过；请发回诊断包。")
print(result_zip)

### 下载
包内包括逐步计时、三组配对比较、逐文档均值、传输量、smoke 检查、GPU 环境、代码与日志。

In [ ]:
files.download(result_zip)